# Network-Voronoi catchments -- full run

Order matters in two places:

1. **Stores are snapped before `adapt_resolution`**, because resolution is chosen
   from the distance to the nearest store and that should be the routable position.
2. **`m1` / `m2` are saved the moment they exist.** They are produced by
   `group_by` + `with_row_index`, and Polars does not guarantee group order across
   runs -- regenerating them later yields *different* node ids than the ones baked
   into `cand` / `dur` / the assignments, and the mismatch fails silently.

Only `snap` and `travel_times` touch OSRM; both cache, so a re-run is cheap.


## 0. Config

In [2]:

import sys, time
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))
from scripts.tdlinx.data_loading import load_geodata

import scripts.isochrones.network_voronoi_2 as nv
import scripts.isochrones.market_metrics as M

# ---- paths -----------------------------------------------------------------
SHARE   = Path(r"X:\Project-Shares\germart-lite\germart-papaio")
PANEL   = SHARE / "panel"
GHSL    = SHARE / r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project_2\data\ghsl_raster_files\ghs_pop_germany_clipped.tif"
WORK    = SHARE / r"edeka_project_2\data\catchments"
OUT     = SHARE / r"edeka_project_2\outputs\catchments" 
CACHE   = SHARE / r"edeka_project_2\outputs\cache" 
CKPT    = SHARE / r"edeka_project_2\outputs\chpt" 

for d in (WORK, CACHE, CKPT, OUT): d.mkdir(parents=True, exist_ok=True)

# ---- parameters ------------------------------------------------------------
OSRM       = "http://localhost:5000"      # or http://192.168.70.123:5000
YEARS      = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
SHOP_TYPES = ["Supermarkt", "Discount Geschäft", "Drogeriemarkt"]
K          = 20                           # 20 certifies rank 1; use 60 if you need ranks 2-3
RANKS      = 3                            # 3 => Huff choice set; 1 => headline catchment only
BETA       = 2.0                          # Huff distance decay -- an assumption, not an estimate
ATTRACT    = "VKFL"                       # Huff attractiveness; set None for time-only
KREIS_ID, KREIS_NAME = "AGS", "GEN"

## 1. Load panel + geodata

In [3]:
panel = pl.read_parquet(PANEL)
gdf, kreis_gdf, bundesland_gdf = load_geodata()

panel = panel.filter(pl.col("Shop Type").is_in(SHOP_TYPES))
print(f"panel: {panel.height:,} rows | {panel['TDLinx'].n_unique():,} stores | "
      f"years {panel['year'].min()}-{panel['year'].max()}")
print("kreis_gdf columns:", list(kreis_gdf.columns))   # confirm KREIS_ID / KREIS_NAME

panel: 447,303 rows | 45,072 stores | years 2014-2025
kreis_gdf columns: ['kreis_key', 'geometry', 'OBJID', 'BEGINN', 'ADE', 'GF', 'BSG', 'LKZ', 'ARS', 'AGS', 'SDV_ARS', 'GEN', 'BEZ', 'IBZ', 'BEM', 'NBD', 'SN_L', 'SN_R', 'SN_K', 'SN_V1', 'SN_V2', 'SN_G', 'FK_S3', 'NUTS', 'ARS_0', 'AGS_0', 'WSK']


In [ ]:
# panel.filter(pl.col("Bundesland") == "Bayern").unique(pl.col("TDLinx"))

## 2. Stores -> snapped

`drop_over_m=300`: a real operating retail location has a road within 300 m. Anything further is a geocode to check, and because a store is a candidate for thousands of cells one bad row corrupts a whole region.

In [4]:
stores = nv.prepare_stores(panel, keep=("Gruppe", "Shop Type"))
stores = nv.snap(stores, base_url=OSRM, cache_path=CACHE, drop_over_m=300)
stores.write_parquet(WORK / "stores_snapped_ALL_YEARS.parquet")
stores.head(3)

prepare_stores: 3,483 store(s) have >1 distinct 'Gruppe' across the panel; most recent kept. Expected for banner switches.
prepare_stores: 1,121 store(s) have >1 distinct 'Shop Type' across the panel; most recent kept. Expected for banner switches.
prepare_stores: 1,830 store(s) move >100m across the panel; median position used. Inspect via coord_span_m.
prepare_stores: 447,303 panel rows -> 45,072 unique stores
  snap: 32/256 chunks
  snap: 64/256 chunks
  snap: 96/256 chunks
  snap: 128/256 chunks
  snap: 160/256 chunks
  snap: 192/256 chunks
  snap: 224/256 chunks
  snap: 256/256 chunks
snap: 45,072 point(s) requested in 246s
  snap distance m: p50=12 p95=44 p99.9=184 max=2245
  dropped 21 point(s) snapping >300m


store_id,lon,lat,x,y,first_year,last_year,coord_span_m,Gruppe,Shop Type,lon_orig,lat_orig,x_orig,y_orig,snap_m
i64,f64,f64,f64,f64,i16,i16,f64,str,str,f64,f64,f64,f64,f64
1565,7.446254,50.420744,4.1395e6,3.0375e6,2014,2025,0.0,"""Rossmann""","""Drogeriemarkt""",7.446,50.421,4.1395e6,3.0375e6,33.71521
1711,10.470959,50.296989,4.3546e6,3.0206e6,2022,2025,0.0,"""Rewe""","""Supermarkt""",10.471,50.297,4.3546e6,3.0207e6,3.168511
5510,9.979978,49.797054,4.3196e6,2.9649e6,2014,2025,0.0,"""Edeka""","""Supermarkt""",9.98,49.797,4.3196e6,2.9649e6,6.211666


In [ ]:
stores_snapped = read_parquet(WORK / "stores_snapped_ALL_YEARS_Supermarkets.parquet")

## 3. Cells -> reduced + snapped

Run once. Every later run reads the parquet and never touches OSRM.

In [5]:

cells_path, map_path = OUT / "cells_snapped_ALL_YEARS.parquet", OUT / "map_full_ALL_YEARS.parquet"

if cells_path.exists() and map_path.exists():
    cells_orig = nv.load_cells(GHSL)
    c2      = pl.read_parquet(cells_path)
    mapping = pl.read_parquet(map_path)
    print(f"loaded {c2.height:,} solve nodes from disk")
else:
    cells_orig = nv.load_cells(GHSL)
    c1, m1 = nv.adapt_resolution(cells_orig, stores)
    c1     = nv.snap(c1, base_url=OSRM, cache_path=CACHE)
    c2, m2 = nv.dedupe(c1, tol_m=0.0)                # 0 = exact, no approximation
    mapping = nv.compose(m1, m2)

    # save immediately: group_by order is not stable across runs
    m1.write_parquet(WORK / "map_adap_ALL_YEARS.parquet")
    m2.write_parquet(WORK / "map_dedupe_ALL_YEARS.parquet")
    mapping.write_parquet(map_path)
    c2.write_parquet(cells_path)

print(f"{cells_orig.height:,} raster cells -> {c2.height:,} solve nodes "
      f"({c2.height/cells_orig.height:.1%})")

load_cells: 7,269,822 populated cells, 83,238,976 people
adapt_resolution: 7,269,822 -> 5,282,194 cells (72.7%); population delta 9.6e-08
  snap: 32/256 chunks
  snap: 64/256 chunks
  snap: 96/256 chunks
  snap: 128/256 chunks
  snap: 160/256 chunks
  snap: 192/256 chunks
  snap: 224/256 chunks
  snap: 256/256 chunks
snap: 5,282,194 point(s) requested in 22862s
  snap distance m: p50=30 p95=201 p99.9=1421 max=12081
  WARNING 71 point(s) failed to snap at all
dedupe(tol=0m): 5,282,194 -> 5,007,599 nodes (94.8%)
7,269,822 raster cells -> 5,007,599 solve nodes (68.9%)


In [6]:
c2, m2 = nv.dedupe(c1, tol_m=0.0)                # 0 = exact, no approximation
c2.write_parquet(cells_path)

dedupe(tol=0m): 5,282,194 -> 5,007,599 nodes (94.8%)


## 4. Preflight

`max_table_size` must be in the thousands (the default 100 rejects every call). If observed speed exceeds `V_MAX_KMH`, the certification bound is unsound -- raise it in the module.

In [ ]:
nv.preflight(c2, base_url=OSRM)

## 5. Travel times

The expensive step. Checkpointed: kill and re-run and it resumes. `progress_every` prints a measured ETA after the first batches.

In [ ]:

matrix_path = OUT / "matrix_ALL_YEARS_Supermarkets.npz"

if matrix_path.exists():
    z    = nv.load_matrix(matrix_path)
    cand, dur = z["candidates"], z["durations"]
    print(f"loaded matrix {dur.shape}")
else:
    cand   = nv.candidates(c2, stores, k=K)
    dur, _ = nv.travel_times(c2, stores, cand, base_url=OSRM,
                                checkpoint=CKPT / "ALL_YEARS_Supermarkets", max_workers=8, progress_every=50)
                                # DELETE PREVIOUS CHECKPOINT FILE BEFORE RUNNING DIFFERENT SPECIFICATION! 
                                # ALMOST CERTAINLY WILL RETURN AN "OUT OF RANGE" ERROR!
    nv.save_matrix(matrix_path, c2, stores, cand, dur)

nv.certification_report(cand, dur, ranks=RANKS)

If rank-1 certification is below ~99%, widen and re-solve only the
uncertified nodes:

```python
cand, dur, ok = nv.refine(c2, stores, cand, dur, k_new=60, rank=1,
                          base_url=OSRM, checkpoint=CKPT / "refine")
nv.save_matrix(matrix_path, c2, stores, cand, dur, cert=ok)
```

## 6. Assign per year + save raster/parquet

`write_label_raster` needs `cells_orig` (it carries row/col) and `mapping`, so the tif comes out at full 100 m resolution regardless of how far the solve was coarsened. The GeoTIFF stores integer codes; the TDLinx lookup is a parquet sidecar.

In [ ]:

assignments = {}
for year in YEARS:
    a = nv.assign(c2, stores, cand, dur, year=year, ranks=RANKS)
    assignments[year] = a

    alive = ((stores["first_year"].to_numpy() <= year)
             & (stores["last_year"].to_numpy() >= year))
    n_orph = nv.orphans(cand, dur, alive).size
    print(f"\n=== {year} ===  {a.filter(pl.col('rank')==1).height:,} nodes assigned, "
          f"{int(alive.sum()):,} stores alive, {n_orph:,} orphan node(s)")
    print(nv.rank_coverage(a))

    nv.catchments(a).write_parquet(OUT / f"catchments_{year}_Supermarkets.parquet")
    a.write_parquet(OUT / f"assignment_{year}_Supermarkets.parquet")
    nv.write_label_raster(cells_orig, a, mapping, GHSL, OUT / f"nearest_store_{year}_Supermarkets.tif")

## 7. Metrics

In [ ]:

labels, lookup = M.rasterize_regions(kreis_gdf, GHSL, cells_orig,
                                     id_col=KREIS_ID, name_col=KREIS_NAME)

metrics = {}
for year in YEARS:
    a  = assignments[year]
    st = stores
    if ATTRACT and ATTRACT in panel.columns:
        st = stores.join(nv.store_year_attrs(panel, year, [ATTRACT]),
                         on="store_id", how="left")
    attract = ATTRACT if (ATTRACT and ATTRACT in st.columns) else None

    alloc  = M.allocate(a, st, rule="huff", attract_col=attract, beta=BETA)
    ca     = M.expand_to_cells(alloc, cells_orig, mapping, labels=labels)
    shares = M.market_shares(ca, st, by="Gruppe")
    conc   = M.hhi(shares)
    local  = M.local_hhi(a, st, attract_col=attract, beta=BETA)
    totals = M.store_totals(ca, st, by="Gruppe")
    metrics[year] = dict(stores=st, alloc=ca, shares=shares, hhi=conc,
                         local=local, totals=totals)

    for name, df in (("shares", shares), ("hhi", conc),
                     ("local_hhi", local), ("gruppe_totals", totals)):
        df.write_parquet(OUT / f"{name}_{year}_Supermarkets.parquet")

## 8. Printed summary

In [ ]:

for year in YEARS:
    m = metrics[year]
    conc, tot, loc = m["hhi"], m["totals"], m["local"]
    named = conc.join(lookup, on="region_code", how="left").filter(
        pl.col("region_code") > 0)

    print(f"\n{'='*68}\n{year}\n{'='*68}")
    print(f"population allocated : {m['alloc']['pop_alloc'].sum():,.0f}")
    print(f"Kreise               : {named.height}")
    print(f"HHI (Gruppe)         : median {named['hhi'].median():,.0f}  "
          f"p10 {named['hhi'].quantile(0.1):,.0f}  p90 {named['hhi'].quantile(0.9):,.0f}")
    print(f"CR3                  : median {named['cr3'].median():.1%}")
    n_hi = named.filter(pl.col("hhi") > 2000).height
    print(f"Kreise above HHI 2000: {n_hi} ({n_hi/named.height:.0%})")
    print(f"local HHI (catchment): median {loc['local_hhi'].median():,.0f}, "
          f"mean rivals {loc['mean_rivals'].mean():.2f}")

    print("\nNational Gruppe shares")
    print(tot.select("Gruppe", pl.col("share").round(4), "pop").head(10))

    print("\nMost concentrated Kreise")
    print(named.select(KREIS_NAME, "hhi", "cr1", "cr3", "n_players", "pop")
               .sort("hhi", descending=True).head(10))

## 9. Maps

In [ ]:

year = YEARS[-2]
m = metrics[year]

fig, axes = plt.subplots(1, 3, figsize=(21, 9))
for ax, (col, cmap, ttl) in zip(axes, [
        ("hhi", "OrRd",  f"HHI (Gruppe), {year}"),
        ("cr1", "PuBu",  f"CR1 -- largest chain's share, {year}"),
        ("n_players", "YlGn", f"Distinct Gruppen present, {year}")]):
    M.plot_choropleth(m["hhi"], kreis_gdf, lookup, id_col=KREIS_ID,
                      value_col=col, cmap=cmap, title=ttl, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:

# National catchment map, coloured by Gruppe
M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                  boundaries=bundesland_gdf, title=f"Nearest-store catchments, {year}")
plt.show()

In [ ]:
panel.filter(pl.col('Gruppe') == "Rewe", pl.col("Shop Type") == "Drogeriemarkt")

In [ ]:

# Zoom to individual Kreise
for name in ["Mannheim", "Heidelberg", "Karlsruhe"]:
    M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                      region=kreis_gdf, region_col=KREIS_NAME, region_name=name,
                      title=f"{name}, {year}")
    plt.show()

In [ ]:
M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                      region=kreis_gdf, region_col=KREIS_NAME, region_name="Berlin",
                      title=f"{name}, {year}")

## 10. Robustness: does the finding survive a different demand model?

Winner-take-all vs Huff, and Huff across a range of decay elasticities. `beta` is an assumption; if the concentration story flips between these, say so before your supervisor asks.

In [ ]:

st = metrics[year]["stores"]
attract = ATTRACT if (ATTRACT and ATTRACT in st.columns) else None
a = assignments[year]

rows = []
specs = [("winner-take-all", dict(rule="wta"))] + \
        [(f"huff beta={b}", dict(rule="huff", attract_col=attract, beta=b))
         for b in (1.5, 2.0, 2.5)]

for label, kw in specs:
    ca = M.expand_to_cells(M.allocate(a, st, **kw), cells_orig, mapping,
                           labels=labels)
    h  = M.hhi(M.market_shares(ca, st, by="Gruppe")).filter(pl.col("region_code") > 0)
    t  = M.store_totals(ca, st, by="Gruppe")
    rows.append({"spec": label,
                 "hhi_median": h["hhi"].median(),
                 "cr3_median": h["cr3"].median(),
                 "share_top1": t["share"][0],
                 "top1": t["Gruppe"][0]})

print(pl.DataFrame(rows))